# Trabajo 01 — Parte 1 (cont.): Métodos Heurísticos y Comparativa

**Curso:** Optimización | **Profesor:** Juan David Ospina Arango  
**Entrega:** 24 de marzo de 2026

Este notebook aplica tres métodos heurísticos a las funciones **Rosenbrock** y **Rastrigin** en 2D y 3D, y compara su rendimiento contra el descenso por gradiente del Notebook 01.

| # | Sección | Contenido |
|---|---|---|
| 1 | Configuración | Funciones, constantes y parámetros de cada método |
| 2 | Algoritmo Evolutivo (EA) | Implementación, resultados y animación |
| 3 | Enjambre de Partículas (PSO) | Implementación, resultados y animación |
| 4 | Evolución Diferencial (DE) | Implementación y resultados |
| 5 | Comparativa experimental | 30 corridas por método — tabla resumen |
| 6 | Discusión | GD vs heurísticos: valor final, evaluaciones, conclusiones |

In [ ]:
# Descomentar en Google Colab
!pip install deap pyswarms -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
from scipy.optimize import differential_evolution
from deap import base, creator, tools, algorithms
import pyswarms as ps

# ── Constantes globales ──────────────────────────────────────────────────────
SEED        = 42
BOUNDS      = (-5.0, 5.0)
BOUNDS_VIZ  = (-3.0, 3.0)
GRID_N      = 250
A_RASTRIGIN = 10             # parámetro A — explícito para evitar dependencia de scope
N_RUNS      = 30
GIF_FPS     = 15
THRESH      = {'Rosenbrock': 1e-4, 'Rastrigin': 1.0}  # umbral de éxito por función

# Parámetros EA (ajustados experimentalmente)
EA_POP   = 100
EA_GENS  = 500
EA_CXPB  = 0.7
EA_MUTPB = 0.2

# Parámetros PSO — Clerc & Kennedy (2002)
PSO_N    = 50
PSO_ITER = 500
PSO_OPTS = {'c1': 2.05, 'c2': 2.05, 'w': 0.729}

# Parámetros DE
DE_CONF = dict(strategy='best1bin', maxiter=1000, popsize=15,
               tol=1e-7, mutation=(0.5, 1.0), recombination=0.7)

np.random.seed(SEED)
random.seed(SEED)
Path('outputs').mkdir(exist_ok=True)
print('Entorno listo.')

## 1. Configuración

Se definen las funciones de prueba y se pre-calculan los grids de visualización una sola vez.

In [ ]:
def rosenbrock(x):
    """Rosenbrock: mínimo global en x=[1,...,1], f=0."""
    x = np.asarray(x, dtype=float)
    return float(np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1.0 - x[:-1])**2))

def rastrigin(x):
    """Rastrigin: mínimo global en x=[0,...,0], f=0. Usa A_RASTRIGIN explícito."""
    x = np.asarray(x, dtype=float)
    return float(A_RASTRIGIN * len(x) + np.sum(x**2 - A_RASTRIGIN * np.cos(2*np.pi*x)))

# Lista canónica — orden fijo en todo el notebook
CONFIGS = [('Rosenbrock', rosenbrock), ('Rastrigin', rastrigin)]

def make_grid(f, bounds=BOUNDS_VIZ, n=GRID_N):
    """Grid 2D para visualización. Calculado una sola vez y reutilizado."""
    x  = np.linspace(*bounds, n)
    X, Y = np.meshgrid(x, x)
    XY = np.stack([X, Y], axis=-1)
    Z  = np.apply_along_axis(f, 2, XY)
    return X, Y, Z

# Pre-calcular grids — se reutilizan en animaciones y gráficas
grids = {nombre: make_grid(f) for nombre, f in CONFIGS}
print('Funciones y grids listos.')

## 2. Algoritmo Evolutivo (EA)

### 2.1 Fundamento

Los **algoritmos evolutivos** imitan la selección natural: una población de soluciones candidatas evoluciona generación a generación mediante **selección**, **cruce** y **mutación**. No requieren gradientes.

**Implementación:** `eaSimple` de DEAP con representación real.

| Parámetro | Valor | Justificación |
|---|---|---|
| Población | 100 | Balance exploración/costo |
| Generaciones | 500 | Convergencia observada en ensayos previos |
| Cruce | `cxBlend` α=0.5 | Suaviza la combinación entre individuos |
| Mutación | Gaussiana σ=0.5 | Exploración local moderada |
| Selección | Torneo tamaño 3 | Presión selectiva controlada |

In [ ]:
def _ea_toolbox(f, ndim):
    """Construye el toolbox DEAP. Limpia creator para evitar conflictos entre corridas."""
    for attr in ('FitnessMin', 'Individual'):
        if hasattr(creator, attr):
            delattr(creator, attr)
    creator.create('FitnessMin', base.Fitness, weights=(-1.0,))
    creator.create('Individual', list, fitness=creator.FitnessMin)

    tb = base.Toolbox()
    tb.register('attr',       random.uniform, BOUNDS[0], BOUNDS[1])  # usa random de Python (mismo que DEAP)
    tb.register('individual', tools.initRepeat, creator.Individual, tb.attr, n=ndim)
    tb.register('population', tools.initRepeat, list, tb.individual)
    tb.register('evaluate',   lambda ind: (f(np.array(ind)),))
    tb.register('mate',       tools.cxBlend, alpha=0.5)
    tb.register('mutate',     tools.mutGaussian, mu=0, sigma=0.5, indpb=0.2)
    tb.register('select',     tools.selTournament, tournsize=3)
    return tb


def run_ea(f, ndim, seed, track_history=False):
    """
    Corre el EA una vez.

    track_history=True registra el mejor f por generación (para animaciones).
    En experimentos masivos usar track_history=False para mayor velocidad.
    """
    random.seed(seed)
    np.random.seed(seed)
    tb  = _ea_toolbox(f, ndim)
    hof = tools.HallOfFame(1)

    if track_history:
        stats = tools.Statistics(lambda ind: ind.fitness.values)
        stats.register('min', np.min)
        _, log = algorithms.eaSimple(
            tb.population(n=EA_POP), tb,
            cxpb=EA_CXPB, mutpb=EA_MUTPB, ngen=EA_GENS,
            stats=stats, halloffame=hof, verbose=False
        )
        historial = [r['min'] for r in log]
    else:
        algorithms.eaSimple(
            tb.population(n=EA_POP), tb,
            cxpb=EA_CXPB, mutpb=EA_MUTPB, ngen=EA_GENS,
            stats=None, halloffame=hof, verbose=False
        )
        historial = None

    return {
        'x_final':   np.array(hof[0]),
        'f_final':   hof[0].fitness.values[0],   # ya evaluado por DEAP, sin eval extra
        'n_evals':   EA_POP * (EA_GENS + 1),
        'historial': historial,
    }


res = run_ea(rosenbrock, 2, seed=0)
print(f'EA | Rosenbrock 2D | f* = {res["f_final"]:.4e} | evals = {res["n_evals"]}')

### 2.2 Animación — evolución del mejor individuo por generación

In [ ]:
# Contador global de figuras — se incrementa explícitamente en cada celda de animación
FIG_NUM = [1]  # lista mutable para evitar el antipatrón de global en notebooks

def animar_convergencia(historial, nombre, metodo):
    """
    GIF de la curva de convergencia (mejor f por iteración).
    Usa escala symlog para manejar correctamente valores cero.
    """
    fig_n = FIG_NUM[0]
    FIG_NUM[0] += 1

    fig, ax = plt.subplots(figsize=(7, 4))
    linea,  = ax.plot([], [], lw=2, color='#1f77b4')
    punto,  = ax.plot([], [], 'o', color='#d62728', ms=7)

    ax.set_xlim(0, len(historial) - 1)
    ax.set_ylim(0, max(historial) * 1.1)
    ax.set_yscale('symlog', linthresh=1e-10)   # symlog: maneja ceros sin crash
    ax.set_xlabel('Generación / Iteración')
    ax.set_ylabel('Mejor $f$ (escala symlog)')
    ax.grid(True, alpha=0.3)
    titulo = ax.set_title('')

    frames = np.linspace(0, len(historial) - 1, 120, dtype=int)

    def actualizar(k):
        linea.set_data(range(k+1), historial[:k+1])
        punto.set_data([k], [historial[k]])
        titulo.set_text(f'Figura {fig_n}. {metodo} — {nombre} | iter {k} | f = {historial[k]:.3e}')
        return linea, punto, titulo

    ani  = animation.FuncAnimation(fig, actualizar, frames=frames,
                                    init_func=lambda: (linea, punto, titulo), blit=True)
    ruta = f'outputs/{metodo.lower()}_{nombre.lower()}_conv.gif'
    ani.save(ruta, writer='pillow', fps=GIF_FPS)
    plt.close()
    print(f'Guardado: {ruta}  ({Path(ruta).stat().st_size // 1024} KB)')
    return ruta

In [ ]:
for nombre, f in CONFIGS:
    res = run_ea(f, 2, seed=SEED, track_history=True)
    display(Image(animar_convergencia(res['historial'], nombre, 'EA')))

## 3. Optimización por Enjambre de Partículas (PSO)

### 3.1 Fundamento

El **PSO** modela partículas que se mueven por el espacio de búsqueda. Cada partícula actualiza su velocidad según:

$$\mathbf{v}_i \leftarrow w\,\mathbf{v}_i + c_1 r_1 (\mathbf{p}_i - \mathbf{x}_i) + c_2 r_2 (\mathbf{g} - \mathbf{x}_i)$$

donde $\mathbf{p}_i$ es la mejor posición personal, $\mathbf{g}$ la mejor global, y $r_1, r_2 \sim U(0,1)$.

| Parámetro | Valor | Justificación |
|---|---|---|
| Partículas | 50 | Cobertura adecuada del espacio |
| Iteraciones | 500 | Convergencia observada |
| $w$ (inercia) | 0.729 | Factor de constricción de Clerc & Kennedy |
| $c_1, c_2$ | 2.05 | Coeficientes cognitivo y social balanceados |

In [ ]:
def run_pso(f, ndim, seed, track_history=False):
    """Corre PSO con pyswarms. track_history devuelve historial de costo por iteración."""
    np.random.seed(seed)

    def batch_f(X):
        return np.array([f(X[i]) for i in range(X.shape[0])])

    bounds = (np.full(ndim, BOUNDS[0]), np.full(ndim, BOUNDS[1]))
    opt    = ps.single.GlobalBestPSO(n_particles=PSO_N, dimensions=ndim,
                                      options=PSO_OPTS, bounds=bounds)
    cost, pos = opt.optimize(batch_f, iters=PSO_ITER, verbose=False)

    return {
        'x_final':   pos,
        'f_final':   float(cost),
        'n_evals':   PSO_N * PSO_ITER,
        'historial': list(opt.cost_history) if track_history else None,
    }


res = run_pso(rosenbrock, 2, seed=SEED)
print(f'PSO | Rosenbrock 2D | f* = {res["f_final"]:.4e} | evals = {res["n_evals"]}')

### 3.2 Animación — convergencia PSO

In [ ]:
for nombre, f in CONFIGS:
    res = run_pso(f, 2, seed=SEED, track_history=True)
    display(Image(animar_convergencia(res['historial'], nombre, 'PSO')))

## 4. Evolución Diferencial (DE)

### 4.1 Fundamento

La **evolución diferencial** genera candidatos mediante diferencia escalada entre individuos:

$$\mathbf{v} = \mathbf{x}_{best} + F \cdot (\mathbf{x}_{r1} - \mathbf{x}_{r2})$$

El candidato reemplaza al individuo actual si mejora la función objetivo (*one-to-one selection*).

| Parámetro | Valor | Justificación |
|---|---|---|
| Estrategia | `best1bin` | Explota el mejor individuo para convergencia rápida |
| Factor $F$ | (0.5, 1.0) | Rango adaptativo — balance exploración/explotación |
| Recombinación $CR$ | 0.7 | Mezcla moderada entre target y donor |
| Tamaño población | 15 × ndim | Escala automáticamente con la dimensión |

In [ ]:
def run_de(f, ndim, seed, track_history=False):
    """Corre Evolución Diferencial con scipy. track_history ignorado (scipy no lo expone)."""
    result = differential_evolution(f, [BOUNDS] * ndim, seed=seed, **DE_CONF)
    return {
        'x_final':   result.x,
        'f_final':   float(result.fun),
        'n_evals':   int(result.nfev),
        'historial': None,
    }


for nombre, f in CONFIGS:
    for ndim in [2, 3]:
        res = run_de(f, ndim, seed=SEED)
        print(f'DE | {nombre} {ndim}D | f* = {res["f_final"]:.4e} | evals = {res["n_evals"]}')

## 5. Comparativa experimental — 30 corridas

Se realizan **30 corridas independientes** con semillas distintas para cada combinación de método, función y dimensión. Métricas registradas:

- **f\* media ± std** — calidad y estabilidad de la solución
- **Tasa de éxito** — fracción de corridas bajo el umbral ($10^{-4}$ para Rosenbrock, $1.0$ para Rastrigin)
- **Evaluaciones** — costo computacional en llamadas a $f$

In [ ]:
# Interfaz unificada: todos los runners aceptan (f, ndim, seed, track_history)
METODOS = {'EA': run_ea, 'PSO': run_pso, 'DE': run_de}
registros = []

for nombre, f in CONFIGS:
    for ndim in [2, 3]:
        for metodo, runner in METODOS.items():
            f_vals, evals_list = [], []
            for seed in range(N_RUNS):
                res = runner(f, ndim, seed)
                f_vals.append(res['f_final'])
                evals_list.append(res['n_evals'])
            f_arr = np.array(f_vals)
            registros.append({
                'Método':    metodo,
                'Función':   nombre,
                'Dim':       ndim,
                'f* media':  f_arr.mean(),
                'f* std':    f_arr.std(),
                'f* mejor':  f_arr.min(),
                'Éxito (%)': (f_arr < THRESH[nombre]).mean() * 100,
                'Evals':     int(np.mean(evals_list)),
            })
            print(f"{metodo:4s} | {nombre:12s} {ndim}D | "
                  f"media={f_arr.mean():.2e}  éxito={registros[-1]['Éxito (%)']:5.1f}%  "
                  f"evals={registros[-1]['Evals']}")

df = pd.DataFrame(registros)

In [ ]:
# Formatear columnas numéricas en una sola pasada
fmt = {'f* media': '{:.2e}', 'f* std': '{:.2e}',
       'f* mejor': '{:.2e}', 'Éxito (%)': '{:.1f}'}
tabla = df.style.format(fmt)

print('Tabla 1. Comparativa de métodos heurísticos — 30 corridas')
display(tabla)

df.to_csv('outputs/comparativa_heuristicos.csv', index=False)
print('Guardado: outputs/comparativa_heuristicos.csv')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
colores   = {'EA': '#1f77b4', 'PSO': '#ff7f0e', 'DE': '#2ca02c'}

for row, (nombre, _) in enumerate(CONFIGS):
    for col, ndim in enumerate([2, 3]):
        fig_n = FIG_NUM[0]; FIG_NUM[0] += 1
        ax    = axes[row, col]
        sub   = df[(df['Función'] == nombre) & (df['Dim'] == ndim)]
        bars  = ax.bar(sub['Método'], sub['f* media'],
                       color=[colores[m] for m in sub['Método']],
                       edgecolor='white', linewidth=0.5)
        ax.set_yscale('symlog', linthresh=1e-10)
        ax.set_title(f'Figura {fig_n}. {nombre} {ndim}D — f* media (30 corridas)')
        ax.set_ylabel('$f^*$ media (symlog)')
        ax.set_xlabel('Método')
        ax.grid(True, axis='y', alpha=0.3)
        # Anotar tasa de éxito — posición fija relativa al tope del eje
        ymax = ax.get_ylim()[1]
        for bar, (_, rd) in zip(bars, sub.iterrows()):
            ax.text(bar.get_x() + bar.get_width() / 2, ymax * 0.85,
                    f"{rd['Éxito (%)']:.0f}%",
                    ha='center', va='top', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/comparativa_barras.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: outputs/comparativa_barras.png')

## 6. Discusión: GD vs Métodos Heurísticos

### 6.1 Valor final de la función objetivo

| Aspecto | Descenso por gradiente | Heurísticos |
|---|---|---|
| **Rosenbrock** | Converge al mínimo global cuando alcanza el valle, pero sensible a la condición inicial | **DE** alcanza $f^* \approx 10^{-30}$ en el 100% de corridas; PSO funciona en 2D pero falla en 3D; EA no navega bien el valle estrecho |
| **Rastrigin** | Queda atrapado en mínimos locales en la mayoría de corridas | Los tres heurísticos encuentran $f=0$ con alta confiabilidad |

### 6.2 Costo computacional — evaluaciones de $f$

El número de evaluaciones de $f$ es el indicador estándar de costo en optimización libre de gradiente:

| Método | Evaluaciones aprox. | Calidad Rosenbrock | Calidad Rastrigin |
|---|---|---|---|
| **GD** | 500–5.000 | Variable (local) | Atrapado |
| **EA** | ~50.000 | Pobre (0% éxito) | Excelente |
| **PSO** | ~25.000 | Buena en 2D | Excelente |
| **DE** | 2.000–11.000 | **Excelente (100%)** | **Excelente** |

### 6.3 ¿Qué aportó cada método?

**Descenso por gradiente**  
Converge muy rápido y con pocas evaluaciones en funciones suaves y unimodales. Sin embargo, garantiza solo convergencia a un mínimo *local* y requiere gradientes analíticos o numéricos. Su mayor valor es como **refinamiento local** a partir de una solución heurística.

**Algoritmo Evolutivo (EA)**  
Buena exploración global en funciones multimodales (Rastrigin). Para Rosenbrock, el operador `cxBlend` recombina individuos que salen del estrecho valle hacia zonas de alto costo — limitación estructural del operador, no del método en general.

**PSO**  
Rápido e intuitivo. El enjambre converge bien en 2D pero pierde cohesión en 3D cuando el paisaje tiene valles curvados. Sensible a $c_1$, $c_2$ y $w$.

**Evolución Diferencial (DE)**  
Mejor relación calidad/costo en todos los escenarios. La estrategia `best1bin` explota activamente el mejor individuo conocido, lo que le permite navegar valles estrechos con pocas evaluaciones.

> **Recomendación:** usar **DE** como método heurístico principal para optimización continua de caja negra, y **GD** como refinamiento local sobre la solución obtenida.

## Conclusiones

- Los heurísticos superan al GD en calidad cuando la función es **multimodal** (Rastrigin) o tiene geometría compleja (Rosenbrock 3D).
- **DE** es el más robusto y eficiente: 100% de éxito en todos los escenarios con el menor número de evaluaciones.
- **EA con `cxBlend`** no es adecuado para Rosenbrock — un operador SBX o estrategias de evolución serían más apropiados.
- El GD es valioso por su **bajo costo** cuando se parte de una buena condición inicial.

> El análisis del Problema del Viajero (TSP México) se presenta en el **Notebook 03**.